# OFR 金融压力指数官方来源预检

只验证每日当前来源覆盖及历史修订说明，保留确切响应、时钟和 NAS 回读；不定义特征或读取 SOXX 结果。

In [1]:
import csv,json,hashlib,subprocess,time,tarfile,tempfile,shlex,zipfile,xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime,timezone,date
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_ofr_financial_stress_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
raw=root/'data/snapshots/stage1_ofr_financial_stress_source_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];checks=[];failure=None
for i,spec in enumerate(plan['exact_requests']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n');r=plan['request'];body=raw/spec['name'];headers=raw/(spec['name']+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(r['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=75);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0);receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>r['max_response_bytes_each']:
  failure='transport_or_size_failure';break
 try:
  if spec['expected']=='text/csv':
   with body.open(newline='',encoding='utf-8-sig') as inp:reader=csv.DictReader(inp);columns=reader.fieldnames;rows=list(reader)
   assert columns and rows and len(rows)>3000
   datecols=[x for x in columns if 'date' in x.lower()];assert len(datecols)==1;dcol=datecols[0]
   
   def parse_day(value):
    value=value.strip()
    try:return date.fromisoformat(value[:10])
    except ValueError:return datetime.strptime(value,'%m/%d/%Y').date()
   dates=[parse_day(row[dcol]) for row in rows];assert len(dates)==len(set(dates))
   numcols=[]
   for col in columns:
    if col==dcol:continue
    good=0
    for row in rows:
     try:float(row[col]);good+=1
     except (TypeError,ValueError):pass
    if good>3000:numcols.append({'column':col,'numeric_rows':good})
   assert numcols and min(dates)<date(2011,1,3) and max(dates)>date(2026,10,7)
   checks.append({'name':spec['name'],'status':'official_CSV_schema_full_span_passed','raw_sha256':receipt['raw_sha256'],'columns':columns,'date_column':dcol,'rows':len(rows),'first_date':str(min(dates)),'last_date':str(max(dates)),'numeric_columns':numcols,'duplicate_dates':0})
  else:
   with zipfile.ZipFile(body) as z:
    names=z.namelist();assert '[Content_Types].xml' in names and 'xl/workbook.xml' in names and any(x.startswith('xl/worksheets/') and x.endswith('.xml') for x in names)
    text_tokens=[]
    for name in names:
     if name.startswith('xl/') and name.endswith('.xml') and (name=='xl/sharedStrings.xml' or name.startswith('xl/worksheets/')):
      element=ET.fromstring(z.read(name));text_tokens.extend([x.text.strip() for x in element.iter() if x.text and x.text.strip() and not x.text.strip().replace('.','',1).isdigit()])
   assert len(text_tokens)>3
   checks.append({'name':spec['name'],'status':'official_revision_workbook_identity_passed','raw_sha256':receipt['raw_sha256'],'zip_member_count':len(names),'nonempty_text_tokens':len(text_tokens),'sample_text_tokens':text_tokens[:15]})
 except Exception as exc:failure='source_identity_or_coverage_failure';checks.append({'name':spec['name'],'status':failure,'error':type(exc).__name__+':'+str(exc)[:180],'raw_sha256':receipt['raw_sha256']});break
 if i+1<len(plan['exact_requests']):time.sleep(r['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='ofr-fsi-source-') as td:
 archive=Path(td)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for x in manifest['files']:t.add(raw/x['name'],arcname='raw/'+x['name'])
  t.add(card/'source_manifest.json',arcname='source_manifest.json');t.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='ofr-fsi-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=120)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as t:
   for x in manifest['files']:
    member=t.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'OFR_FSI_official_current_full_span_and_revision_document_received_PIT_pending' if failure is None and len(checks)==2 else 'bounded_OFR_FSI_source_partial_or_failed','failure':failure,'GET_attempts':state['GET_attempts'],'checks':checks,'NAS_archive_sha256':digest,'NAS_remote_and_scoped_restore_verified':True,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'historical_first_public_vintage_qualified':False,'historical_methodology_revisions_qualified':False};(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'failure':failure,'GETs':state['GET_attempts'],'checks':checks,'NAS_sha256':digest})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'OFR_FSI_official_current_full_span_and_revision_document_received_PIT_pending', 'failure': None, 'GETs': 2, 'checks': [{'name': 'fsi.csv', 'status': 'official_CSV_schema_full_span_passed', 'raw_sha256': 'c59b58d64c9d63e4684071baf8ff54b127158f76025638634df91a49fbea89e9', 'columns': ['Date', 'OFR FSI', 'Credit', 'Equity valuation', 'Safe assets', 'Funding', 'Volatility', 'United States', 'Other advanced economies', 'Emerging markets'], 'date_column': 'Date', 'rows': 6784, 'first_date': '2000-01-03', 'last_date': '2026-10-20', 'numeric_columns': [{'column': 'OFR FSI', 'numeric_rows': 6784}, {'column': 'Credit', 'numeric_rows': 6784}, {'column': 'Equity valuation', 'numeric_rows': 6784}, {'column': 'Safe assets', 'numeric_rows': 6784}, {'column': 'Funding', 'numeric_rows': 6784}, {'column': 'Volatility', 'numeric_rows': 6784}, {'column': 'United States', 'numeric_rows': 6784}, {'column': 'Other advanced economies', 'numeric_rows': 6784}, {'column': 'Emerging markets', 'numeric_

# OFR 金融压力指数：官方全历史来源与修订边界

官方 CSV 收到 6,784 个无重复日期（2000-01-03 至 2026-10-20），`Funding` 等九列均有数值；官方修订工作簿与 CSV 原字节已归档至 NAS，远端哈希及每个原始成员限定恢复通过。0 SOXX 结果读取、0 拟合、0 账户。

来源风险是实质性的：2018-05-18 的 158 个改值日期、2018-12-04 的 2 个日期，以及 2023-06-27 的 382 个日期均有官方旧/新值；2023 年替换资金类别变量后，Funding 绝对变动中位数 0.307、最大 0.764 个指数点。工作簿“新值”与当前 CSV 在对应日期逐项一致，不能把它当作当年旧值。官网另提示 2026 年数据订正，旧版本不在 2023 工作簿内。

[OFR 于 2017-10-25 发布的说明](https://www.financialresearch.gov/from-the-director/2017/10/25/the-ofr-introduces-financial-stability-index-and-financial-vulnerabilities-monitor/)把现行每日 FSI 列为新工具；[2014 年 OFR 报告](https://www.financialresearch.gov/annual-reports/files/office-of-financial-research-annual-report-2014.pdf)描述较早的 FSI，但不能证明现行 33 指标序列的 2000–2017 回填与当时版本完全相同。因此**不能**把今天 CSV 的 2011–2017 数值直接并入冻结 Stage1 全训练期当点时信号。

[当前 OFR 页面](https://www.financialresearch.gov/financial-stress-index/)称公布值较观测日滞后两个工作日；这不是各历史日的首次接收证明。下一步只可对 2018–2022 的 10:32 恢复接收器做单独的条件性来源映射，保守滞后、复原官方已列修订旧值、保留未知；通过后还须检验旧 21 输入增量和完整 0bp 账户。

来源 NAS SHA-256 `074e2c56f6f95a4f56382a32e25c65ad560e5a53ee6e4bb7fd58e50bc247b369`。本卡决定：`official_full_numeric_history_received_pre2017_PIT_and_revision_boundaries_unqualified`，不授予交易时点、SOXX 超额或晋升资格。